In [ ]:
import kagglehub
path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
print(f"path:\n{path}")

Using Colab cache for faster access to the 'new-plant-diseases-dataset' dataset.
path:
/kaggle/input/new-plant-diseases-dataset


In [ ]:
import numpy as np
import os
import subprocess
import pickle
from PIL import Image
from numpy.lib.stride_tricks import sliding_window_view

class Relu:
    def activate(x):
        return x *(x >=0)
    def back(x):
        return (x > 0)
class Sigmoid:
    def activate(x):
        return(1 / (1 + np.exp(-x)))
    def back(outp):
        return(outp *(1-outp))
class Tanh:
    def activate(x):
        return np.tanh(x)
    def back(outp):
        return (1-(outp**2))

class Softmax:
    def activate(x):
        tmp = np.exp(x)
        outp /=np.sum(tmp)

    def back(outp , tr):
        return ((outp-tr) /len(tr) )

class Softmax1:
    @staticmethod
    def activate(x):
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum(axis=0)

    @staticmethod
    def back(pred, target):
        return (pred - target)


In [ ]:
def send_alert(message):
  print("[+]" , message)
def send_to_server(a):
  print("[!]" , a)


In [ ]:
def import_imgs(img_name : list , dir = "" , is_gray = True , right_ans = 0)->list:
    ans =list()
    if is_gray:
        for i in img_name:
            ans.append( (right_ans , np.array(Image.open(dir+i).convert('L').resize((128, 128)))/255))
        return ans
    else:
        for i in img_name:
            ans.append( right_ans, np.array(Image.open(dir+i)))
        return ans


In [ ]:
import numpy as np
import os
import subprocess
import pickle
from PIL import Image
from numpy.lib.stride_tricks import sliding_window_view
from joblib import Parallel, delayed


def learning1 (dataset : list , alfa , rounds , R_w = [], w_w =[] , size_of_r = 3 , w_w_sizes = [0 , 5000 , 10] , nrm = [0.0001 , 0.01] , r_size = 2):
    out_side = 128 - size_of_r + 1
    w_w_sizes[0] = (out_side**2) * r_size
    if(len(R_w) == 0):
        print("[+]init R_w ..." , end="")
        std_dev = np.sqrt(2.0 / size_of_r**2)
        for i in range (r_size):
            R_w.append(np.random.randn(size_of_r**2, 1) * std_dev)
        print("\r[+]init w_w ..." , end="")

        for i in range (1 , len(w_w_sizes)):
            w_w.append(np.random.randn(w_w_sizes[i-1] , w_w_sizes[i]) * nrm[1])
        print("\r[+]init w inited")


    for round in range(rounds):
        j = 0
        alll_l = len(dataset)
        r = 0
        for answer , data in dataset:
            print(f"\r progress : {(j/alll_l)*100} \t%" , flush=True , end = "")
            j+=1
            bbff1 = np.hstack(R_w)
            windows1 = sliding_window_view(data, (size_of_r, size_of_r))
            flat_windows = windows1.reshape(-1, size_of_r**2)
            all_res = np.dot(flat_windows, bbff1)

            h_out, w_out = windows1.shape[:2]
            l0_1 = all_res.T.reshape(len(R_w), h_out, w_out)
            buff = l0_1.flatten()

            l1 = Relu.activate(buff.flatten().dot(w_w[0]))
            raw_pred = l1.dot(w_w[1])

            pred = Softmax1.activate(raw_pred)
            h = np.argmax(pred)
            if(h == answer):
                r+=1

            if(np.isnan(pred).any()):
                print("[+]NAN DETECTED")
                import sys
                sys.exit(0)
                exit()
            need = np.zeros(w_w_sizes[2])
            need[answer] = 1
            delta = Softmax1.back(pred, need)
            d1 = delta.dot(w_w[1].T)* Relu.back(l1)
            d_flattened = d1.dot(w_w[0].T)
            d_map = d_flattened.reshape(l0_1.shape)
            for i in range(r_size):
                grad_R = np.tensordot(d_map[i], windows1, axes=((0,1), (0,1)))
                R_w[i] -= alfa[0] * grad_R.reshape(-1, 1)
            w_w[1]-=alfa[1]*np.outer(l1 , delta)
            w_w[0]-=alfa[1]*np.outer(buff.flatten() , d1 )
        send_to_server(str(r / len(dataset)))
    return R_w , w_w

def tests_d (dataset : list , alfa , rounds , R_w = [], w_w =[] , size_of_r = 3 , w_w_sizes = [0 , 5000 , 10] , nrm = [0.0001 , 0.01] , r_size = 2, layout_data = True , layout_names = [] ):
    if(len(R_w) == 0):
        print("[-] Ws is empty ")

    for round in range(rounds):
        j = 0
        alll_l = len(dataset)
        r = 0
        for answer , data in dataset:
            if(not layout_data):
              print(f"\r progress : {(j/alll_l)*100} \t%" , flush=True , end = "")
            j+=1
            bbff1 = np.hstack(R_w)
            windows1 = sliding_window_view(data, (size_of_r, size_of_r))
            flat_windows = windows1.reshape(-1, size_of_r**2)
            all_res = np.dot(flat_windows, bbff1)

            h_out, w_out = windows1.shape[:2]
            l0_1 = all_res.T.reshape(len(R_w), h_out, w_out)
            buff = l0_1.flatten()
            l1 = Relu.activate(buff.flatten().dot(w_w[0]))
            raw_pred = l1.dot(w_w[1])

            pred = Softmax1.activate(raw_pred)
            h = np.argmax(pred)
            if(h == answer):
                r+=1
            if(layout_data):
              anss = np.argsort(pred)[::-1]
              print(f"[!]righ answer : {layout_names[answer]}")
              for i in range(5):
                print(f"\t1.{layout_names[anss[i]]}")
        send_to_server(str(r / len(dataset)))
    return R_w , w_w

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
folder = "/content/drive/MyDrive/PlantWeights"
if not os.path.exists(folder):
        os.makedirs(folder)

def save_1(R_w , w_w , pck):
  weights_data = {
    'R_w': R_w,
    'w_w': w_w
  }
  with open(f'weights{pck}.pkl', 'wb') as f:
    pickle.dump(weights_data, f)

  print(f"Веса успешно сохранены в weights{pck}.pkl")
def save_2( name , data , pth ="./"):
  with open(pth + name , 'wb') as f:
    pickle.dump(data, f)

  print(f"Веса успешно сохранены в {name}.pkl")
def load_1(name : str):
  with open(name , 'rb') as f:
    wd = pickle.load(f)
    R_w = wd['R_w']
    w_w = wd['w_w']
    return R_w , w_w

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#main
import os
import random

plants = []
dir = path+"/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid"

plants_all = os.listdir(dir)

print(plants_all)


for i in plants_all:
  a = i
  if(a not in plants):
    plants.append(a)
print(plants)
R_w , w_w = load_1(folder+"/we.pkl")
packs = 10
s_pack = 0
'''for i1 in range(packs *s_pack  , 1600 , packs):
  pck = int(i1/packs)
  print(f"pack : {pck}")
  dataset = list()
  for i in plants_all:
    all_pics = os.listdir(dir+"/"+i)[i1 : i1+packs]
    answer = plants.index(i)
    bfff = (dir+"/"+i+"/")
    #print(bfff)
    dataset+=import_imgs(all_pics , dir=bfff , right_ans=answer)
  random.shuffle(dataset)
  print(f"[+]start pack {pck} \t leigh {len(dataset)}")

  R_w , w_w =learning1(dataset=dataset , alfa = [0.0005 , 0.0005] , rounds= 1 , w_w_sizes=[0 , 500 , len(plants)] , R_w=R_w , w_w=w_w , r_size = 8)
  #save_1(R_w , w_w , pck)
  weights_data = {
    'R_w': R_w,
    'w_w': w_w
  }
  save_2(f"weights{pck}.pkl" , weights_data , pth = folder+"/")
weights_data = {
    'R_w': R_w,
    'w_w': w_w
}

# Сохраняем в файл (wb — write binary, запись в бинарном режиме)
with open('weights.pkl', 'wb') as f:
    pickle.dump(weights_data, f)
'''
dataset =list()
print("[/]INIT DATASET ..." , end= "")
for i1 in range(packs *s_pack  , 10 , packs):
  for i in plants_all:
    all_pics = os.listdir(dir+"/"+i)[i1 : i1+packs]
    print( i , all_pics[0])
    answer = plants.index(i)
    bfff = (dir+"/"+i+"/")
    #print(bfff)
    dataset+=import_imgs(all_pics , dir=bfff , right_ans=answer)
  # random.shuffle(dataset)
print("\r[+]DATASET INITED ..." )

tests_d(dataset=dataset , alfa = [0.0005 , 0.0005] , rounds= 1 , w_w_sizes=[0 , 500 , len(plants)] , R_w=R_w , w_w=w_w , r_size = 8 , layout_names =plants  , layout_data = 1)

print("Веса успешно сохранены в weights.pkl")



['Tomato___Late_blight', 'Tomato___healthy', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Potato___healthy', 'Corn_(maize)___Northern_Leaf_Blight', 'Tomato___Early_blight', 'Tomato___Septoria_leaf_spot', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Strawberry___Leaf_scorch', 'Peach___healthy', 'Apple___Apple_scab', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Bacterial_spot', 'Apple___Black_rot', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Peach___Bacterial_spot', 'Apple___Cedar_apple_rust', 'Tomato___Target_Spot', 'Pepper,_bell___healthy', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Potato___Late_blight', 'Tomato___Tomato_mosaic_virus', 'Strawberry___healthy', 'Apple___healthy', 'Grape___Black_rot', 'Potato___Early_blight', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Common_rust_', 'Grape___Esca_(Black_Measles)', 'Raspberry___healthy', 'Tomato___Leaf_Mold', 'Tomato__

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')